In [1]:
from pathlib import Path

import pandas as pd

# Step 1: Load this repo's processed generator-level data
base_dir = Path('..')
dispatchable = pd.read_csv(base_dir / 'data/processed/dispatchable_generator_hourly_dataset.csv', parse_dates=['timestamp'])
renewable = pd.read_csv(base_dir / 'data/processed/renewable_generator_hourly_dataset.csv', parse_dates=['timestamp'])

dispatchable_frame = dispatchable.rename(
    columns={
        'timestamp': 'Timestamp',
        'capability_mw': 'Capability',
        'actual_output_mw': 'Output',
        'fuel_type': 'FuelType',
    }
)[['Timestamp', 'generator', 'FuelType', 'Capability', 'Output']]

renewable_frame = renewable.rename(
    columns={
        'timestamp': 'Timestamp',
        'available_capacity': 'Capability',
        'output': 'Output',
        'fuel_type': 'FuelType',
    }
)[['Timestamp', 'generator', 'FuelType', 'Capability', 'Output']]

all_generators = pd.concat([dispatchable_frame, renewable_frame], ignore_index=True)
all_generators['FuelType'] = all_generators['FuelType'].str.upper()

cap = all_generators.pivot_table(index='Timestamp', columns='generator', values='Capability', aggfunc='first').sort_index()
out = all_generators.pivot_table(index='Timestamp', columns='generator', values='Output', aggfunc='first').sort_index()

gen_to_tech = all_generators[['generator', 'FuelType']].drop_duplicates().sort_values('generator').reset_index(drop=True)

In [2]:
cap.head()

generator,ABKENORA,ADELAIDE,AGUASABON,ALEXANDER,AMARANTH,AMHERST ISLAND,APIROQUOIS,ARMOW,ARNPRIOR,ATIKOKAN-G1,...,WELLS,WEST LINCOLN NRWF,WESTWINDSOR,WHITBYCGS,WHITEDOG,WINDSOR AIRPORT SF,WOLFE ISLAND,YORKCGS-G1,YORKCGS-G2,ZURICH
Timestamp,,,,,,,,,,,,,,,,,,,,,
2019-05-01 00:00:00,NaN,60.0,NaN,NaN,199.0,1.0,NaN,180.0,NaN,NaN,...,NaN,230.0,NaN,NaN,NaN,50.0,180.0,NaN,NaN,99.0
2019-05-01 01:00:00,NaN,60.0,NaN,NaN,199.0,5.0,NaN,180.0,NaN,NaN,...,NaN,230.0,NaN,NaN,NaN,50.0,180.0,NaN,NaN,99.0
2019-05-01 02:00:00,NaN,60.0,NaN,NaN,199.0,10.0,NaN,180.0,NaN,NaN,...,NaN,230.0,NaN,NaN,NaN,50.0,180.0,NaN,NaN,99.0
2019-05-01 03:00:00,NaN,60.0,NaN,NaN,199.0,10.0,NaN,180.0,NaN,NaN,...,NaN,230.0,NaN,NaN,NaN,50.0,180.0,NaN,NaN,99.0
2019-05-01 04:00:00,NaN,60.0,NaN,NaN,199.0,5.0,NaN,180.0,NaN,NaN,...,NaN,230.0,NaN,NaN,NaN,50.0,180.0,NaN,NaN,99.0


In [3]:
gen_to_tech = gen_to_tech.rename(columns={'GeneratorName': 'generator', 'fuel_type': 'FuelType'}).copy()
gen_to_tech['FuelType'] = gen_to_tech['FuelType'].str.upper()
gen_to_tech_dict = dict(zip(gen_to_tech['generator'], gen_to_tech['FuelType']))
print(gen_to_tech_dict)

{'ABKENORA': 'HYDRO', 'ADELAIDE': 'WIND', 'AGUASABON': 'HYDRO', 'ALEXANDER': 'HYDRO', 'AMARANTH': 'WIND', 'AMHERST ISLAND': 'WIND', 'APIROQUOIS': 'HYDRO', 'ARMOW': 'WIND', 'ARNPRIOR': 'HYDRO', 'ATIKOKAN-G1': 'BIOFUEL', 'AUBREYFALLS': 'HYDRO', 'BARRETT': 'HYDRO', 'BECK1': 'HYDRO', 'BECK2': 'HYDRO', 'BECK2 PGS': 'HYDRO', 'BELLE RIVER': 'WIND', 'BLAKE': 'WIND', 'BORNISH': 'WIND', 'BOW LAKE': 'WIND', 'BOW LAKE 2': 'WIND', 'BRIGHTON BEACH': 'GAS', 'BRUCEA-G1': 'NUCLEAR', 'BRUCEA-G2': 'NUCLEAR', 'BRUCEA-G3': 'NUCLEAR', 'BRUCEA-G4': 'NUCLEAR', 'BRUCEB-G5': 'NUCLEAR', 'BRUCEB-G6': 'NUCLEAR', 'BRUCEB-G7': 'NUCLEAR', 'BRUCEB-G8': 'NUCLEAR', 'CALSTOCKGS': 'BIOFUEL', 'CAMERONFALLS': 'HYDRO', 'CANYON': 'HYDRO', 'CARDINAL': 'GAS', 'CARIBOUFALLS': 'HYDRO', 'CARMICHAEL': 'HYDRO', 'CEDAR POINT 2': 'WIND', 'CHATSFALLS': 'HYDRO', 'CHENAUX': 'HYDRO', 'CLERGUE': 'HYDRO', 'COCHRANECGS': 'GAS', 'COMBER': 'WIND', 'CRYSLER': 'WIND', 'DA WATSON': 'HYDRO', 'DARLINGTON-G1': 'NUCLEAR', 'DARLINGTON-G2': 'NUCLEAR', 

### Clean generator df's
(ie; HALTONHILLS-LT.G3 vs HALTONHILLS-LT_G3)

In [4]:
def verify_same_generators():
    # Get generator names from each source
    cap_gens = set(cap.columns)
    out_gens = set(out.columns)
    tech_gens = set(gen_to_tech['generator'])

    # Generators in all three files
    common_gens = cap_gens & out_gens & tech_gens

    # Total number in each
    num_cap = len(cap_gens)
    num_out = len(out_gens)
    num_tech = len(tech_gens)
    num_common = len(common_gens)

    # Generators not present in all three
    cap_only = cap_gens - common_gens
    out_only = out_gens - common_gens
    tech_only = tech_gens - common_gens

    # Report
    print("🔍 Generator Matching Summary")
    print(f"Total in capability file:      {num_cap}")
    print(f"Total in output file:          {num_out}")
    print(f"Total in gen_to_tech mapping:  {num_tech}")
    print(f"In all three files:          {num_common}")
    print(f"Not in tech: {len(cap_only)}")

    print("Generators in cap/out only:", cap_only)
    print("Generators in tech only:", tech_only)

verify_same_generators()

🔍 Generator Matching Summary
Total in capability file:      181
Total in output file:          181
Total in gen_to_tech mapping:  181
In all three files:          181
Not in tech: 0
Generators in cap/out only: set()
Generators in tech only: set()


In [5]:
## Manually created from research on generators missing in the generator_to_type.csv file
missing_generator_type_map = {
    'NPCOCHRANE': 'GAS',
    'TADOUGLAS': 'GAS',
}

'''
Drop from out and cap as generators 
-ONEIDA ENERGY STORAGE is a battery storage not a generator so drop to simplify model
-THUNDERBAY-G3 was demolished in 2021 no longer in use
'''
drop_generators_dict = {'ONEIDA ENERGY STORAGE', 'THUNDERBAY-G3'}


## Manually created from comparison of files (perfect overlap between old and new names in timeseries)
rename_map = {
    'HALTONHILLS-LT_G1': 'HALTONHILLS-LT.G1',
    'HALTONHILLS-LT_G2': 'HALTONHILLS-LT.G2',
    'HALTONHILLS-LT_G3': 'HALTONHILLS-LT.G3',
    'RAILBEDWF-LT_AG_SR': 'RAILBEDWF-LT.AG_SR',
    'MCLEANSMTNWF-LT_AG_T1': 'MCLEANSMTNWF-LT.AG_T1',
    'SANDUSK-LT_AG_T1': 'SANDUSK-LT.AG_T1'
    }

def check_timestamp_overlap(df, old_name, new_name):
    if old_name not in df.columns or new_name not in df.columns:
        return None  # One or both names not in DataFrame

    old_times = df[old_name].dropna().index
    new_times = df[new_name].dropna().index
    overlap = old_times.intersection(new_times)
    return len(overlap)

results = []

for old, new in rename_map.items():
    overlap_cap = check_timestamp_overlap(cap, old, new)
    overlap_out = check_timestamp_overlap(out, old, new)
    
    results.append({
        'Old Name': old,
        'New Name': new,
        'Cap Overlap': overlap_cap,
        'Out Overlap': overlap_out,
        'Valid Rename': (overlap_cap == 0 and overlap_out == 0)
    })

df_rename_check = pd.DataFrame(results)
df_rename_check



,Old Name,New Name,Cap Overlap,Out Overlap,Valid Rename
0,HALTONHILLS-LT_G1,HALTONHILLS-LT.G1,None,None,False
1,HALTONHILLS-LT_G2,HALTONHILLS-LT.G2,None,None,False
2,HALTONHILLS-LT_G3,HALTONHILLS-LT.G3,None,None,False
3,RAILBEDWF-LT_AG_SR,RAILBEDWF-LT.AG_SR,None,None,False
4,MCLEANSMTNWF-LT_AG_T1,MCLEANSMTNWF-LT.AG_T1,None,None,False
5,SANDUSK-LT_AG_T1,SANDUSK-LT.AG_T1,None,None,False


In [6]:
# 1. Merge renamed columns in cap and out
def merge_columns(df, rename_map):
    for old, new in rename_map.items():
        if old in df.columns and new in df.columns:
            # Merge values: take non-null values from old, overwrite nulls in new
            df[new] = df[new].combine_first(df[old])
            df.drop(columns=old, inplace=True)
        elif old in df.columns:
            # If new doesn't exist, just rename old to new
            df.rename(columns={old: new}, inplace=True)
    return df

cap = merge_columns(cap, rename_map)
out = merge_columns(out, rename_map)

# 2. Drop unnecessary generators
cap.drop(columns=drop_generators_dict.intersection(cap.columns), inplace=True, errors='ignore')
out.drop(columns=drop_generators_dict.intersection(out.columns), inplace=True, errors='ignore')

# 3. Update the generator-to-tech dictionary directly
# First, recreate the base dictionary from the cleaned DataFrame
gen_to_tech_dict = dict(zip(gen_to_tech['generator'], gen_to_tech['FuelType']))

# Now add missing generators directly to the dictionary
gen_to_tech_dict.update(missing_generator_type_map)

In [7]:
## Verify all generators in out/cap have a "type"
verify_same_generators()


🔍 Generator Matching Summary
Total in capability file:      181
Total in output file:          181
Total in gen_to_tech mapping:  181
In all three files:          181
Not in tech: 0
Generators in cap/out only: set()
Generators in tech only: set()


In [8]:
def trim_generator_data(df):
    trimmed_df = df.copy()
    for col in df.columns:
        series = df[col]
        first_valid = series.first_valid_index()
        last_valid = series.last_valid_index()
        if first_valid is not None and last_valid is not None:
            # Set NaN only *before* and *after* valid data
            trimmed_df.loc[trimmed_df.index < first_valid, col] = pd.NA
            trimmed_df.loc[trimmed_df.index > last_valid, col] = pd.NA
    return trimmed_df

cap = trim_generator_data(cap)
cap = cap.apply(pd.to_numeric, errors='coerce')

out = trim_generator_data(out)
out = out.apply(pd.to_numeric, errors='coerce')

cap.head()

generator,ABKENORA,ADELAIDE,AGUASABON,ALEXANDER,AMARANTH,AMHERST ISLAND,APIROQUOIS,ARMOW,ARNPRIOR,ATIKOKAN-G1,...,WELLS,WEST LINCOLN NRWF,WESTWINDSOR,WHITBYCGS,WHITEDOG,WINDSOR AIRPORT SF,WOLFE ISLAND,YORKCGS-G1,YORKCGS-G2,ZURICH
Timestamp,,,,,,,,,,,,,,,,,,,,,
2019-05-01 00:00:00,NaN,60.0,NaN,NaN,199.0,1.0,NaN,180.0,NaN,NaN,...,NaN,230.0,NaN,NaN,NaN,50.0,180.0,NaN,NaN,99.0
2019-05-01 01:00:00,NaN,60.0,NaN,NaN,199.0,5.0,NaN,180.0,NaN,NaN,...,NaN,230.0,NaN,NaN,NaN,50.0,180.0,NaN,NaN,99.0
2019-05-01 02:00:00,NaN,60.0,NaN,NaN,199.0,10.0,NaN,180.0,NaN,NaN,...,NaN,230.0,NaN,NaN,NaN,50.0,180.0,NaN,NaN,99.0
2019-05-01 03:00:00,NaN,60.0,NaN,NaN,199.0,10.0,NaN,180.0,NaN,NaN,...,NaN,230.0,NaN,NaN,NaN,50.0,180.0,NaN,NaN,99.0
2019-05-01 04:00:00,NaN,60.0,NaN,NaN,199.0,5.0,NaN,180.0,NaN,NaN,...,NaN,230.0,NaN,NaN,NaN,50.0,180.0,NaN,NaN,99.0


# Paramaters Estimation

## P_max for each generator

In [9]:
# Compute P_max from capability DataFrame
p_max = cap.max()

# Convert to DataFrame
df_p_max = pd.DataFrame({'P_max': p_max})

# Optional: ensure the generator names are the index
df_p_max.index.name = 'generator'

# Preview the result
df_p_max.head()


,P_max
generator,
ABKENORA,11.0
ADELAIDE,60.0
AGUASABON,49.0
ALEXANDER,65.0
AMARANTH,199.0


## P_Min for each generator

In [10]:
import numpy as np
# 1) Replace 0s with NaN to ignore them in min calculation
out_nonzero = out.replace(0, np.nan)

# 2) Compute minimum non-zero output per generator
min_nonzero_output = out_nonzero.min()

# 3) Assemble into a DataFrame
df_min_output = pd.DataFrame({
    'P_min': min_nonzero_output,
})

# Optional: Ensure index order matches original generators
df_min_output = df_min_output.loc[out.columns]

df_min_output.head()

,P_min
generator,
ABKENORA,7.0
ADELAIDE,1.0
AGUASABON,1.0
ALEXANDER,1.0
AMARANTH,1.0


## T_up and T_down (minimum time up and down after changing state)


### Estimating $T^U$ and $T^D$ from Historical Generator Output

To assign realistic minimum up-time ($T^U$) and down-time ($T^D$) values for each generator, Analyzed historical hourly output data using the following steps:

---

#### 1. Binary On/Off Classification

Each generator’s output was normalized by its $P^{\max}_g$.
Marked a generator as **on** if its output was ≥ 5% of $P^{\max}_g$; otherwise, it was considered **off**.

---

#### 2. Run-Length Extraction

Parsed each generator’s time series to detect contiguous **on-runs** and **off-runs**, recording their durations.
Runs that began or ended at the edge of the dataset were treated as **censored**.

---

#### 3. Kaplan–Meier Estimation

Used Kaplan–Meier survival curves to model how long generators typically stay on or off.
Defined $T^U$ and $T^D$ as the **5th percentile** of on/off run durations—i.e., the shortest duration seen in at least 95% of cases.

---

#### 4. Outlier Filtering & Tech-Level Smoothing

To avoid overfitting rare behaviors:

* Generators with **too few cycles** (e.g., <30 on-runs) were assigned **tech-level median** values.
* Others kept their individual estimates.

---

#### Final Output

Each generator is assigned a validated $T^U$ and $T^D$, grounded in empirical behavior and smoothed by technology when needed. These values are used directly in the model’s minimum up/down time constraints.



In [11]:
import pandas as pd

# 1) Define on/off as strictly greater than 0
df_onoff = (out > 0).astype(int)  # 1 if output > 0, else 0

# (Optional) Remove this if you're no longer using threshold-based logic
# threshold = 0.05  # Not needed anymore
# p_max_series = p_max if isinstance(p_max, pd.Series) else pd.Series(p_max, index=out.columns)
# on_threshold_mw = p_max_series * threshold

# 2) Function to extract consecutive on/off runs
def extract_runs(binary_series):
    """
    Returns two lists: on_runs and off_runs,
    each a list of (run_length, is_censored)
    """
    runs_on = []
    runs_off = []
    curr_val = binary_series.iloc[0]
    run_len = 1
    for i in range(1, len(binary_series)):
        if binary_series.iloc[i] == curr_val:
            run_len += 1
        else:
            is_censored = False
            if i == len(binary_series) - 1 and curr_val == 1:
                is_censored = True
            if curr_val == 1:
                runs_on.append((run_len, is_censored))
            else:
                runs_off.append((run_len, is_censored))
            curr_val = binary_series.iloc[i]
            run_len = 1
    # Add final run
    if curr_val == 1:
        runs_on.append((run_len, True))
    else:
        runs_off.append((run_len, False))
    return runs_on, runs_off

# 3) Apply run extraction to each generator
on_runs_per_gen = {}
off_runs_per_gen = {}

for gen in df_onoff.columns:
    s = df_onoff[gen].dropna()
    on_runs, off_runs = extract_runs(s)
    on_runs_per_gen[gen] = on_runs
    off_runs_per_gen[gen] = off_runs


In [12]:
import pandas as pd
import numpy as np

# Step 4: KM survival estimator (custom)
def compute_km_survival(durations, event_observed):
    df = pd.DataFrame({'duration': durations, 'event': event_observed})
    df = df.sort_values('duration')
    times = sorted(df['duration'].unique())
    surv = 1.0
    survival = {}
    for t in times:
        d = df[(df['duration'] == t) & (df['event'] == 1)].shape[0]
        at_risk = df[df['duration'] >= t].shape[0]
        if at_risk > 0:
            surv *= (1 - d / at_risk)
        survival[t] = surv
    return survival

# Step 5: Compute T_up and T_down for each generator
results = []
for gen in df_onoff.columns:
    # On-run
    on_runs = on_runs_per_gen.get(gen, [])
    durations_up = [length for length, cens in on_runs]
    events_up    = [int(not cens) for length, cens in on_runs]
    surv_up = compute_km_survival(durations_up, events_up) if durations_up else {}
    T_up = next((t for t, s in surv_up.items() if s <= 0.95), None)
    
    # Off-run
    off_runs = off_runs_per_gen.get(gen, [])
    durations_down = [length for length, cens in off_runs]
    events_down    = [int(not cens) for length, cens in off_runs]
    surv_down = compute_km_survival(durations_down, events_down) if durations_down else {}
    T_down = next((t for t, s in surv_down.items() if s <= 0.95), None)
    
    # Add number of on-runs
    num_on_runs = len(on_runs)
    
    results.append({
        'generator': gen,
        'T_up (hrs)': T_up,
        'T_down (hrs)': T_down,
        'num_on_runs': num_on_runs  # <-- Added
    })

# Step 6: Display results
df_results = pd.DataFrame(results).set_index('generator')

# Cut out any estimates that didnt have enough cycle data
min_cycles = 10
valid = df_results['num_on_runs'] >= min_cycles
df_results.loc[~valid, ['T_up (hrs)', 'T_down (hrs)']] = np.nan
df_results.drop(columns=["num_on_runs"], inplace=True)


In [13]:
pd.set_option("display.max_rows", None)

print(df_results)

                                   T_up (hrs)  T_down (hrs)
generator                                                  
ABKENORA                                  NaN           NaN
ADELAIDE                                  1.0           1.0
AGUASABON                                 3.0           1.0
ALEXANDER                                 NaN           NaN
AMARANTH                                  1.0           1.0
AMHERST ISLAND                            1.0           1.0
APIROQUOIS                                NaN           NaN
ARMOW                                     1.0           1.0
ARNPRIOR                                  2.0           1.0
ATIKOKAN-G1                               6.0           4.0
AUBREYFALLS                               1.0           1.0
BARRETT                                   1.0           1.0
BECK1                                     NaN           NaN
BECK2                                     NaN           NaN
BECK2 PGS                               

In [14]:
# Step 1: Map each generator to its fuel type
df_results['FuelType'] = df_results.index.map(gen_to_tech_dict)

# Step 2: Compute FuelType-level medians
tech_median_up = df_results.groupby('FuelType')['T_up (hrs)'].median()
tech_median_down = df_results.groupby('FuelType')['T_down (hrs)'].median()

# Step 3: Define adjustment logic (NaNs or extreme outliers get replaced with median)
def adjust_t(row, col, tech_medians):
    val = row[col]
    median = tech_medians.loc[row['FuelType']]
    
    if pd.isna(val):
        return median
    elif val > 1.5 * median:
        return median
    else:
        return val

# Step 4: Apply adjusted values to new columns
df_results['T_up (hrs)'] = df_results.apply(
    lambda r: adjust_t(r, 'T_up (hrs)', tech_median_up), axis=1
)

df_results['T_down (hrs)'] = df_results.apply(
    lambda r: adjust_t(r, 'T_down (hrs)', tech_median_down), axis=1
)

print(df_results)

                                   T_up (hrs)  T_down (hrs) FuelType
generator                                                           
ABKENORA                                  2.0           1.0    HYDRO
ADELAIDE                                  1.0           1.0     WIND
AGUASABON                                 3.0           1.0    HYDRO
ALEXANDER                                 2.0           1.0    HYDRO
AMARANTH                                  1.0           1.0     WIND
AMHERST ISLAND                            1.0           1.0     WIND
APIROQUOIS                                2.0           1.0    HYDRO
ARMOW                                     1.0           1.0     WIND
ARNPRIOR                                  2.0           1.0    HYDRO
ATIKOKAN-G1                               1.0           1.0  BIOFUEL
AUBREYFALLS                               1.0           1.0    HYDRO
BARRETT                                   1.0           1.0    HYDRO
BECK1                             

## Max Up, Max Down

To estimate each generator’s ramp-up and ramp-down limits, we analyzed hourly output data by calculating the change in production between consecutive hours. We considered only the hours where a generator increased or decreased its output, ignoring flat or zero changes.

To reduce the influence of noise or one-off spikes, we used the 95th percentile of all observed positive ramps for each generator. This gives a realistic high-end ramp rate without being overly conservative.

After estimating those empirical ramps, we apply the same physical overrides used in the main UC model:

- nuclear units are set to **60% of rated power per hour**
- hydro units are allowed to move from **0 to Pmax within the hour**

These final ramp rates are used directly in the model’s hourly ramping constraints to ensure operational realism.




In [15]:
import numpy as np
import pandas as pd

# 1) Compute hour-to-hour diffs
diffs = out.diff()

# 2) Extract only true up-ramps (drop zeros & negatives)
diffs_up = diffs.where(diffs > 0, np.nan)

# 3) Compute 95th percentile of those positive ramps
ramp_up = diffs_up.quantile(0.95)

# 4) Likewise for down-ramps: flip sign, keep only negatives
diffs_down = diffs.where(diffs < 0, np.nan)
ramp_down = (-diffs_down).quantile(0.95)

# 5) Combine into a DataFrame
df_ramps = pd.DataFrame({
    'Ramp Up (MW/hr)':   ramp_up,
    'Ramp Down (MW/hr)': ramp_down
})

# Ensure the order matches `out.columns`
df_ramps = df_ramps.loc[out.columns]

# Apply the physical overrides used in the main UC model
fuel_type = pd.Series(gen_to_tech_dict).reindex(df_ramps.index).str.upper()
p_max_series = df_p_max['P_max'].reindex(df_ramps.index)

nuclear_mask = fuel_type.eq('NUCLEAR')
hydro_mask = fuel_type.eq('HYDRO')

df_ramps.loc[nuclear_mask, 'Ramp Up (MW/hr)'] = 0.60 * p_max_series[nuclear_mask]
df_ramps.loc[nuclear_mask, 'Ramp Down (MW/hr)'] = 0.60 * p_max_series[nuclear_mask]

df_ramps.loc[hydro_mask, 'Ramp Up (MW/hr)'] = p_max_series[hydro_mask]
df_ramps.loc[hydro_mask, 'Ramp Down (MW/hr)'] = p_max_series[hydro_mask]

df_ramps.head()


,Ramp Up (MW/hr),Ramp Down (MW/hr)
generator,,
ABKENORA,11.0,11.0
ADELAIDE,20.0,19.0
AGUASABON,49.0,49.0
ALEXANDER,65.0,65.0
AMARANTH,31.0,30.0


## Startup Cost and Shutdown Cost

In [16]:
import pandas as pd

# Assumed inputs as DataFrames:
# 1) p_min_df: DataFrame with a column 'P_min', indexed by generator ID
# 2) ramp_df:  DataFrame with columns 'Ramp Up (MW/hr)' and 'Ramp Down (MW/hr)', indexed by generator ID
# 3) C_V:      dict or Series mapping technology → variable cost ($/MWh)
# 4) gen_to_tech: dict mapping generator ID → technology key

C_V = {
    'GAS': 6.3364,
    'NUCLEAR': 2.852,
    'HYDRO': 0,
    'WIND': 0,
    'SOLAR': 0,
    'BIOFUEL': 5.208
}

# Extract Series from DataFrames
p_min = df_min_output['P_min']
p_max = df_p_max['P_max']
p_reg = (p_min + p_max) / 2 


ramp_up = df_ramps['Ramp Up (MW/hr)']
ramp_down = df_ramps['Ramp Down (MW/hr)']

# 1) Compute implied startup/shutdown durations (hours)
T_startup  = p_reg / ramp_up
T_shutdown = p_reg / ramp_down

# 2) Compute energy consumed during ramp (MWh)
#    Assume linear ramp: average output = P_min / 2
E_startup  = 0.5 * p_reg * T_startup
E_shutdown = 0.5 * p_reg * T_shutdown

# 3) Compute costs ($)
startup_cost = pd.Series(index=p_reg.index, dtype=float)
shutdown_cost = pd.Series(index=p_reg.index, dtype=float)
var_cost = pd.Series(index=p_reg.index, dtype=float)

for gen in p_reg.index:
    tech = gen_to_tech_dict[gen]
    Cv   = C_V[tech]
    startup_cost[gen]  = Cv * E_startup.loc[gen]
    shutdown_cost[gen] = Cv * E_shutdown.loc[gen]
    var_cost[gen] = Cv

# 4) Combine into one DataFrame for clarity
cost_df = pd.DataFrame({
    'T_startup (h)':      T_startup,
    'T_shutdown (h)':     T_shutdown,
    'E_startup (MWh)':    E_startup,
    'E_shutdown (MWh)':   E_shutdown,
    'Cost_startup ($)':   startup_cost,
    'Cost_shutdown ($)':  shutdown_cost,
    'Cost_variable ($/MWh)': var_cost
})


cost_df


/var/folders/gc/2hjqw94d0fg3h32mgnj49r7m0000gn/T/ipykernel_57590/886492729.py:44: RuntimeWarning: invalid value encountered in scalar multiply
  startup_cost[gen]  = Cv * E_startup.loc[gen]
/var/folders/gc/2hjqw94d0fg3h32mgnj49r7m0000gn/T/ipykernel_57590/886492729.py:45: RuntimeWarning: invalid value encountered in scalar multiply
  shutdown_cost[gen] = Cv * E_shutdown.loc[gen]


,T_startup (h),T_shutdown (h),E_startup (MWh),E_shutdown (MWh),Cost_startup ($),Cost_shutdown ($),Cost_variable ($/MWh)
generator,,,,,,,
ABKENORA,0.818182,0.818182,3.681818,3.681818,0.000000,0.000000,0.0000
ADELAIDE,1.525000,1.605263,23.256250,24.480263,0.000000,0.000000,0.0000
AGUASABON,0.510204,0.510204,6.377551,6.377551,0.000000,0.000000,0.0000
ALEXANDER,0.507692,0.507692,8.376923,8.376923,0.000000,0.000000,0.0000
AMARANTH,3.225806,3.333333,161.290323,166.666667,0.000000,0.000000,0.0000
AMHERST ISLAND,1.562500,1.704545,29.296875,31.960227,0.000000,0.000000,0.0000
APIROQUOIS,0.546610,0.546610,17.628178,17.628178,0.000000,0.000000,0.0000
ARMOW,1.740385,1.774510,78.752404,80.296569,0.000000,0.000000,0.0000
ARNPRIOR,0.506098,0.506098,10.501524,10.501524,0.000000,0.000000,0.0000


## Variable Cost

In [17]:
C_V = {
    'GAS': 6.3364,
    'NUCLEAR': 2.852,
    'HYDRO': 0,
    'WIND': 0,
    'SOLAR': 0,
    'BIOFUEL': 5.208
}

## Generator Commision date

In [18]:
def get_commission_year_df(df):
    """
    Given a DataFrame with timestamps as index and generator names as columns,
    returns a DataFrame with each generator's commission year.
    """
    commission_years = {}
    for gen in df.columns:
        first_valid = df[gen].first_valid_index()
        if first_valid is not None:
            commission_years[gen] = first_valid.year
        else:
            commission_years[gen] = None
    return pd.DataFrame.from_dict(commission_years, orient='index', columns=['commission_year'])

commission_year_df = get_commission_year_df(cap)
print(commission_year_df)

                                   commission_year
ABKENORA                                      2025
ADELAIDE                                      2019
AGUASABON                                     2025
ALEXANDER                                     2025
AMARANTH                                      2019
AMHERST ISLAND                                2019
APIROQUOIS                                    2025
ARMOW                                         2019
ARNPRIOR                                      2025
ATIKOKAN-G1                                   2025
AUBREYFALLS                                   2025
BARRETT                                       2025
BECK1                                         2025
BECK2                                         2025
BECK2 PGS                                     2025
BELLE RIVER                                   2019
BLAKE                                         2019
BORNISH                                       2019
BOW LAKE                       

In [19]:
import pandas as pd

# If any DataFrame uses a 'generator' column instead of index:
for df in [df_p_max, df_min_output, df_results, df_ramps, cost_df, commission_year_df]:
    if 'generator' in df.columns:
        df.set_index('generator', inplace=True)

# 1) Combine all parameter DataFrames along columns
df_all_params = pd.concat([df_p_max, df_min_output, df_results, df_ramps, cost_df, commission_year_df], axis=1)
df_all_params.index.name = 'Generator'

# 3) Inspect the combined DataFrame
df_all_params

# 4) Save to CSV
output_path = "../Data/GeneratorParamaters.csv"
df_all_params.to_csv(output_path)

print(f"Combined parameters saved to {output_path}")


Combined parameters saved to ../Data/GeneratorParamaters.csv
